<hr>

# 🧫 DATA CLEANING - Valeurs Foncieres


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### 📂 IMPORTs

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200) # to display all columns in the dataframe
print("✅ Libraries imported successfully")

✅ Libraries imported successfully


<hr>

## 0️⃣ AGGREGATE DATASETS

<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

- 6 TXT FILES INTO 1 CSV
- Dropped not used columns

### fixed text accents & added INSEE 
- dep_code  =  Code departement
- com_code  =  Code commune
- street_id =  Code voie

- No more messy text thanks to:
    - encoding latin-1
    - decoding utf-8

### new life

In [4]:
import pandas as pd

# -------------------------
# FILES
# -------------------------
dvf_files = [
    "../data/raw/ValeursFoncieres-2020-S2.txt",
    "../data/raw/ValeursFoncieres-2021.txt",
    "../data/raw/ValeursFoncieres-2022.txt",
    "../data/raw/ValeursFoncieres-2023.txt",
    "../data/raw/ValeursFoncieres-2024.txt",
    "../data/raw/ValeursFoncieres-2025-S1.txt"
]

# -------------------------
# COLUMNS TO KEEP
# -------------------------
usecols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Commune",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nombre de lots",
    "Surface terrain",
    "No voie",
    "Code voie",
    "B/T/Q",
    "Voie",
    "Code postal",
    "Code departement",
    "Code commune"
]

# -------------------------
# OUTPUT
# -------------------------
output_file = "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv"

# -------------------------
# DTYPES ON READ
# -------------------------
dtype_dict = {
    "Date mutation": str,
    "Nature mutation": str,
     # "Valeur fonciere": float,
    "No voie": str,
    "Code voie": str,
    "B/T/Q": str,
    "Voie": str,
    "Code postal": str,
    "Commune": str,
    "Code departement": str,
    "Code commune": str,
     # "Nombre de lots": int,
    "Type local": str,
     # "Surface reelle bati": float,
     # "Nombre pieces principales": int,
     # "Surface terrain": float
}

# -------------------------
# TEXT COLUMNS TO FIX
# -------------------------
text_cols = [
    "Nature mutation",
    "Voie",
    "Commune",
    "Type local"
]

# -------------------------
# ENCODING FIX
# -------------------------
def fix_encoding(series):
    series = series.copy()
    mask = series.notna()
    series.loc[mask] = (
        series.loc[mask]
        .str.encode("latin-1", errors="ignore")
        .str.decode("utf-8", errors="ignore")
        .str.strip()
    )
    return series

# -------------------------
# CLEAN STRING CODES
# removes trailing .0 before converting/keeping as string
# -------------------------
def clean_code_str(series, zfill=None):
    series = series.copy()
    mask = series.notna()

    series.loc[mask] = (
        series.loc[mask]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

    series = series.astype("string")

    if zfill is not None:
        mask2 = series.notna()
        series.loc[mask2] = series.loc[mask2].str.zfill(zfill)

    return series

# -------------------------
# PROCESSING
# -------------------------
with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for file in dvf_files:
        print(f"Processing: {file}")

        for chunk in pd.read_csv(
            file,
            sep="|",
            usecols=usecols,
            chunksize=50_000,
            encoding="latin-1",
            decimal=",",
            dtype=dtype_dict,
            low_memory=False
        ):
            # -------------------------
            # FIX TEXT ENCODING
            # -------------------------
            for col in text_cols:
                chunk[col] = fix_encoding(chunk[col])

            # -------------------------
            # CLEAN "No voie" -> string, no .0
            # -------------------------
            chunk["No voie"] = clean_code_str(chunk["No voie"])

            # -------------------------
            # CLEAN POSTAL CODE -> string, 5 digits
            # -------------------------
            chunk["Code postal"] = clean_code_str(chunk["Code postal"], zfill=5)

            # -------------------------
            # CLEAN DEPARTMENT CODE -> string
            # 01 to 95  -> 2 digits (mainland France)
            # 2A / 2B stay unchanged (mainland Corsica)
            # 971 to 976 -> 3 digits (overseas departments)
            
            # -------------------------
            chunk["Code departement"] = clean_code_str(chunk["Code departement"])

            mask_dep = chunk["Code departement"].notna()

            # Corsica codes stay as-is
            mask_corsica = chunk["Code departement"].isin(["2A", "2B"])

            # Overseas departments: 971, 972, 973, 974, 976 -> 3 digits
            mask_overseas = chunk["Code departement"].str.startswith("97", na=False)

            # Metropolitan numeric departments except Corsica -> 2 digits
            mask_numeric = chunk["Code departement"].str.fullmatch(r"\d+", na=False)
            mask_metro = mask_dep & mask_numeric & ~mask_overseas & ~mask_corsica

            chunk.loc[mask_metro, "Code departement"] = (
                chunk.loc[mask_metro, "Code departement"].str.zfill(2)
            )

            chunk.loc[mask_overseas, "Code departement"] = (
                chunk.loc[mask_overseas, "Code departement"].str.zfill(3)
            )

            # -------------------------
            # CLEAN COMMUNE CODE -> string
            # 01 to 95, 2A, 2B -> 3 digits
            # 97#               -> 2 digits
            # -------------------------
            chunk["Code commune"] = clean_code_str(chunk["Code commune"])

            mask_com = chunk["Code commune"].notna()
            mask_97 = chunk["Code departement"].str.startswith("97", na=False)

            chunk.loc[mask_com & mask_97, "Code commune"] = (
                chunk.loc[mask_com & mask_97, "Code commune"].str.zfill(2)
            )
            chunk.loc[mask_com & ~mask_97, "Code commune"] = (
                chunk.loc[mask_com & ~mask_97, "Code commune"].str.zfill(3)
            )

            # ----------------------------------
            # CLEAN VOIE CODE -> string, no .0
            # ----------------------------------
            chunk["Code voie"] = clean_code_str(chunk["Code voie"])

            # -------------------------
            # KEEP TEXT KEYS AS STRING
            # -------------------------
            chunk["Voie"] = chunk["Voie"].astype("string")
            chunk["Commune"] = chunk["Commune"].astype("string")
            chunk["Nature mutation"] = chunk["Nature mutation"].astype("string")
            chunk["B/T/Q"] = chunk["B/T/Q"].astype("string")
            chunk["Code postal"] = chunk["Code postal"].astype("string")
            chunk["Code departement"] = chunk["Code departement"].astype("string")
            chunk["Code commune"] = chunk["Code commune"].astype("string")
            chunk["Code voie"] = chunk["Code voie"].astype("string")
            chunk["No voie"] = chunk["No voie"].astype("string")


            # -------------------------
            # CREATE FULL INSEE CODE
            # -------------------------
            # INSEE format:
            # - Mainland (01–95, 2A, 2B): insee_code = DD + CCC  → 5 characters
            # - Overseas (971–976):       insee_code = DDD + CC → 5 characters
            # 
            # Note:
            # - Code departement already cleaned (01–95, 2A/2B, 971–976)
            # - Code commune already formatted:
            #     * 3 digits for mainland
            #     * 2 digits for overseas
            # 
            # So we can safely concatenate

            #mask_insee = chunk["Code departement"].notna() & chunk["Code commune"].notna()

            #chunk["insee_code"] = (
            #    chunk["Code departement"]
            #    .astype("string")
            #    .str.cat(chunk["Code commune"].astype("string"))
            #)

            # ensure missing values remain <NA>
            #chunk.loc[~mask_insee, "insee_code"] = pd.NA

            # -------------------------
            # NUMERIC CONVERSION
            # -------------------------
            chunk["Valeur fonciere"] = pd.to_numeric(chunk["Valeur fonciere"], errors="coerce")
            chunk["Surface reelle bati"] = pd.to_numeric(chunk["Surface reelle bati"], errors="coerce")
            chunk["Surface terrain"] = pd.to_numeric(chunk["Surface terrain"], errors="coerce")

            chunk["Nombre de lots"] = (
                pd.to_numeric(chunk["Nombre de lots"], errors="coerce")
                .round()
                .astype("Int64")
            )
            chunk["Nombre pieces principales"] = (
                pd.to_numeric(chunk["Nombre pieces principales"], errors="coerce")
                .round()
                .astype("Int64")
            )

            # -------------------------
            # MEMORY OPTIMIZATION
            # -------------------------
            chunk["Type local"] = chunk["Type local"].astype("category")

            # -------------------------
            # WRITE CLEAN UTF-8 CSV
            # -------------------------
            chunk.to_csv(
                f_out,
                index=False,
                header=not header_written,
                encoding="utf-8"
            )

            header_written = True
            del chunk

print("✅ Encoding fixed + CSV saved in clean UTF-8")
print(f"✅ CSV File saved at {output_file}")

Processing: ../data/raw/ValeursFoncieres-2020-S2.txt
Processing: ../data/raw/ValeursFoncieres-2021.txt
Processing: ../data/raw/ValeursFoncieres-2022.txt
Processing: ../data/raw/ValeursFoncieres-2023.txt
Processing: ../data/raw/ValeursFoncieres-2024.txt
Processing: ../data/raw/ValeursFoncieres-2025-S1.txt
✅ Encoding fixed + CSV saved in clean UTF-8
✅ CSV File saved at ../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv


### **DISPLAY and EXPLORATION**

In [13]:
import pandas as pd

# -------------------------
# DTYPES ON LOAD
# -------------------------
load_dtypes = {
    "Date mutation": "string",
    "Nature mutation": "string",
    "No voie": "string",
    "B/T/Q": "string",
    "Code voie": "string",
    "Voie": "string",
    "Code postal": "string",
    "Commune": "string",
    "Code departement": "string",
    "Code commune": "string",
    "Type local": "string"
}
# ---------------------------------------------------------------------------
# Load the cleaned CSV file with specified dtypes and UTF-8 encoding
# ---------------------------------------------------------------------------
df = pd.read_csv(
    "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv",
    dtype=load_dtypes,
    encoding="utf-8")

# ------------------------------
# DISPLAY & EXPLORATION 
# ------------------------------
# numeric columns
df["Valeur fonciere"] = pd.to_numeric(df["Valeur fonciere"], errors="coerce")
df["Surface reelle bati"] = pd.to_numeric(df["Surface reelle bati"], errors="coerce")
df["Surface terrain"] = pd.to_numeric(df["Surface terrain"], errors="coerce")
df["Nombre de lots"] = pd.to_numeric(df["Nombre de lots"], errors="coerce").astype("Int64")
df["Nombre pieces principales"] = pd.to_numeric(df["Nombre pieces principales"], errors="coerce").astype("Int64")

# Display df DataFrame
print("RAW Valeurs Foncieres:")
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")
display(df.head())

# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())
print(100 * "-")

# Display unique values of each column in df
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")


# ---------------------------------------------
# Define the Renaming Dictionary
# ---------------------------------------------
rename_columns = {
    "Date mutation": "transaction_date",
    "Nature mutation": "transaction_type",
    "Valeur fonciere": "property_value",
    "No voie": "street_number",
    "B/T/Q": "btq_code",
    "Voie": "street_name",
    "Code postal": "postal_code",
    "Commune": "com_name",
    "Nombre de lots": "number_of_lots",
    "Type local": "property_type",
    "Surface reelle bati": "building_real_surface",
    "Nombre pieces principales": "main_rooms_count",
    "Surface terrain": "land_surface",
    "Code commune" : "com_code",
    "Code departement" : "dep_code",
    "Code voie" : "street_code"
}

# ---------------------------------------------
# Rename columns using the rename Dictionary
# ---------------------------------------------
df.rename(columns=rename_columns, inplace=True)

# --------------------------------
# Save dataset in a new CSV file
# --------------------------------
# Save cleaned DataFrame to a new CSV file
df.to_csv("../data/processed/INTERIM_00_TRANSFORM_ValeursFoncieres.csv", index=False, encoding="utf-8")
print("✅ Best Raw DataFrame saved to INTERIM_00_TRANSFORM_ValeursFoncieres.csv")


RAW Valeurs Foncieres:
Dataset Shape: 20102739 rows and 16 columns



,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Code voie,Voie,Code postal,Commune,Code departement,Code commune,Nombre de lots,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain
0,01/07/2020,Vente,31234.16,<NA>,<NA>,B064,SAINT JULIEN,01560,SAINT-JULIEN-SUR-REYSSOUZE,01,367,0,<NA>,NaN,<NA>,1192.0
1,01/07/2020,Vente,278000.00,<NA>,<NA>,B188,A LA PEROUSE,01250,CORVEISSIAT,01,125,0,<NA>,NaN,<NA>,10092.0
2,01/07/2020,Vente,278000.00,<NA>,<NA>,B188,A LA PEROUSE,01250,CORVEISSIAT,01,125,0,<NA>,NaN,<NA>,4570.0
3,01/07/2020,Vente,278000.00,<NA>,<NA>,B079,AUX COMMUNS,01250,CORVEISSIAT,01,125,0,<NA>,NaN,<NA>,5750.0
4,01/07/2020,Vente,278000.00,<NA>,<NA>,B033,EN COMBARNAUD,01250,SIMANDRE-SUR-SURAN,01,408,0,<NA>,NaN,<NA>,648170.0


Data Types & Missing Values of Each Column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20102739 entries, 0 to 20102738
Data columns (total 16 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Date mutation              string 
 1   Nature mutation            string 
 2   Valeur fonciere            float64
 3   No voie                    string 
 4   B/T/Q                      string 
 5   Code voie                  string 
 6   Voie                       string 
 7   Code postal                string 
 8   Commune                    string 
 9   Code departement           string 
 10  Code commune               string 
 11  Nombre de lots             Int64  
 12  Type local                 string 
 13  Surface reelle bati        float64
 14  Nombre pieces principales  Int64  
 15  Surface terrain            float64
dtypes: Int64(2), float64(3), string(11)
memory usage: 2.4 GB


None

----------------------------------------------------------------------------------------------------
➡️ Date mutation (1806) 🧾 unique values:
<StringArray>
['01/07/2020', '04/07/2020', '02/07/2020', '07/07/2020', '03/07/2020',
 '09/07/2020', '06/07/2020', '08/07/2020', '10/07/2020', '15/07/2020',
 ...
 '01/01/2025', '16/03/2025', '27/04/2025', '09/03/2025', '13/04/2025',
 '06/04/2025', '23/03/2025', '01/05/2025', '18/05/2025', '05/01/2025']
Length: 1806, dtype: string
----------------------------------------------------------------------------------------------------
➡️ Nature mutation (6) 🧾 unique values:
<StringArray>
[                             'Vente', 'Vente en l'état futur d'achèvement',
              'Vente terrain à bâtir',                            'Echange',
                       'Adjudication',                      'Expropriation']
Length: 6, dtype: string
----------------------------------------------------------------------------------------------------
➡️ Valeur fonci

In [ ]:
import pandas as pd

# -------------------------
# DTYPES ON LOAD
# -------------------------
load_dtypes = {
    "Date mutation": "string",
    "Nature mutation": "string",
    "No voie": "string",
    "B/T/Q": "string",
    "Code voie": "string",
    "Voie": "string",
    "Code postal": "string",
    "Commune": "string",
    "Code departement": "string",
    "Code commune": "string",
    "Type local": "string"
}
# ---------------------------------------------------------------------------
# Load the cleaned CSV file with specified dtypes and UTF-8 encoding
# ---------------------------------------------------------------------------
df = pd.read_csv(
    "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv",
    dtype=load_dtypes,
    encoding="utf-8")

<hr>

## 📕 NOTES TO DO


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

['01' nan '02' '03' '04' '05' '06' '07' '08' '09' '10' '11' '12' '13' '14'
 '15' '16' '17' '18' '19' '21' '22' '23' '24' '25' '26' '27' '28' '29'
 '2A' '2B' '30' '31' '32' '33' '34' '35' '36' '37' '38' '39' '40' '41'
 '42' '43' '44' '45' '46' '47' '48' '49' '50' '51' '52' '53' '54' '55'
 '56' '57' '58' '59' '60' '61' '62' '63' '64' '65' '66' '67' '68' '69'
 '70' '71' '72' '73' '74' '75' '76' '77' '78' '79' '80' '81' '82' '83'
 '84' '85' '86' '87' '88' '89' '90' '91' '92' '93' '94' '95' '971' '972' '973' '974' '976']

#### ADAPT MY MAIN DATASET
COLUMN: dep_code:
- nan in "dep_code."Is it the missing code '96'?
- incosistent values in 

from: '97#' and '98#' in my df (main dataset)
to: '971' '972' '973' '974' '976'

USE commun COLUMN: "com_name" between df and df_com_dep dataframes:
- add "insee_code", "dep_code", "dep_name","com_type" from df_com_dep
- and if there is no match fix the names that aren't consistent in df

// fix invalid values
- other columns FIXED
- com_name in df
- dep_code in df_com_dep
- add from df_com_dep  to df the columns 
	- insee_code
	- com_type
	- dep_code
	- dep_name

// convert type
- street_number to object
- main_rooms_count to int

// deal with nulls
- fill nulls with 'Unknown' & '0' or drop
- generate calculated value_per_m2
	- property_value/(sum surfaces)

// deal with duplicates
- create transaction_id

// deal with outliers
- assess if there are any outliers
- drop those transactions rows


<hr>

## 1️⃣ STANDARDIZE NAMES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

| Original Column Name          |      Standardized        | Description                                      |
| ----------------------------- | ------------------------ |------------------------------------------------- |
| **Date mutation**             | `transaction_date`       | Date when the property was sold                  |
| **Nature mutation**           | `transaction_type`       | Nature of transaction (sale, donation, etc.)     |
| **Valeur fonciere**           | `property_value`         | Transaction price in euros                       |
| **No voie**                   | `street_number`          | Street number of the property                    |
| **B/T/Q**                     | `btq_code`               | Bâtiment/Type/Quartier code for identification   |
| **Voie**                      | `street_name`            | Full street name                                 |
| **Code postal**               | `postal_code`            | Postal code (categorical, not numeric)           |
| **Commune**                   | `com_name`               | Commune name                                     |
| **Nombre de lots**            | `number_of_lots`         | Number of lots in the property                   |
| **Type local**                | `property_type`          | Property type (Apartment, House, etc.)           |
| **Surface reelle bati**       | `building_real_surface`  | Built area in square meters                      |
| **Nombre pieces principales** | `main_rooms_count`       | Number of main rooms (living / bedrooms)         |
| **Surface terrain**           | `land_surface`           | Land area in square meters                       |
| **Code departement**           | `dep_code`           | DD department code                       |
| **Code commune**           | `com_code`           | CCC commune code                       |
| **Code voie**           | `street_id`           | street id for extracting street name if needed                       |


### **SHAPE & HEAD**

In [9]:
import pandas as pd
df = pd.read_csv("../data/processed/RAW_ValeursFoncieres.csv")

# Display df DataFrame
print("RAW Valeurs Foncieres:")
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")
display(df.head())

C:\Users\sboub\AppData\Local\Temp\ipykernel_17920\916945881.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/RAW_ValeursFoncieres.csv")


RAW Valeurs Foncieres:
Dataset Shape: 20102739 rows and 15 columns



,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Voie,Code postal,Commune,Code departement,Code commune,Nombre de lots,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain
0,01/07/2020,Vente,31234.16,NaN,NaN,SAINT JULIEN,01560,SAINT-JULIEN-SUR-REYSSOUZE,1,367,0,NaN,NaN,NaN,1192.0
1,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,01250,CORVEISSIAT,1,125,0,NaN,NaN,NaN,10092.0
2,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,01250,CORVEISSIAT,1,125,0,NaN,NaN,NaN,4570.0
3,01/07/2020,Vente,278000.00,NaN,NaN,AUX COMMUNS,01250,CORVEISSIAT,1,125,0,NaN,NaN,NaN,5750.0
4,01/07/2020,Vente,278000.00,NaN,NaN,EN COMBARNAUD,01250,SIMANDRE-SUR-SURAN,1,408,0,NaN,NaN,NaN,648170.0


### **INFO & UNIQUE VALUES**

In [10]:
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())
print(100 * "-")

# Display unique values of each column in df
# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")

Data Types & Missing Values of Each Column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20102739 entries, 0 to 20102738
Data columns (total 15 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Date mutation              object 
 1   Nature mutation            object 
 2   Valeur fonciere            float64
 3   No voie                    float64
 4   B/T/Q                      object 
 5   Voie                       object 
 6   Code postal                object 
 7   Commune                    object 
 8   Code departement           object 
 9   Code commune               int64  
 10  Nombre de lots             int64  
 11  Type local                 object 
 12  Surface reelle bati        float64
 13  Nombre pieces principales  float64
 14  Surface terrain            float64
dtypes: float64(5), int64(2), object(8)
memory usage: 2.2+ GB


None

----------------------------------------------------------------------------------------------------
➡️ Date mutation (1806) 🧾 unique values:
['01/07/2020' '04/07/2020' '02/07/2020' ... '01/05/2025' '18/05/2025'
 '05/01/2025']
----------------------------------------------------------------------------------------------------
➡️ Nature mutation (6) 🧾 unique values:
['Vente' "Vente en l'état futur d'achèvement" 'Vente terrain à bâtir'
 'Echange' 'Adjudication' 'Expropriation']
----------------------------------------------------------------------------------------------------
➡️ Valeur fonciere (466668) 🧾 unique values:
[3.123416e+04 2.780000e+05 3.985000e+03 ... 7.088090e+05 5.503861e+05
 2.441758e+07]
----------------------------------------------------------------------------------------------------
➡️ No voie (9127) 🧾 unique values:
[  nan  347. 1084. ... 8562. 8301. 8423.]
----------------------------------------------------------------------------------------------------
➡️ B/T/Q 

### **DESCRIBE & COUNT**

In [11]:
import pandas as pd
# display statistics of the dataframe
display(df.describe())
# display count of non-null values in each column
display(df.count())

,Valeur fonciere,No voie,Code commune,Nombre de lots,Surface reelle bati,Nombre pieces principales,Surface terrain
count,1.990835e+07,1.262823e+07,2.010274e+07,2.010274e+07,1.187762e+07,1.187762e+07,1.370604e+07
mean,1.541231e+06,7.065920e+02,2.067871e+02,4.351320e-01,6.685673e+01,1.817896e+00,2.873696e+03
std,1.684142e+07,2.006358e+03,1.664372e+02,8.438582e-01,6.413538e+02,2.085038e+00,1.879976e+04
min,1.000000e-02,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.870000e+04,8.000000e+00,7.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,2.490000e+02
50%,1.650000e+05,2.500000e+01,1.710000e+02,0.000000e+00,3.500000e+01,1.000000e+00,6.250000e+02
75%,3.052900e+05,9.900000e+01,2.950000e+02,1.000000e+00,8.200000e+01,3.000000e+00,1.796000e+03
max,1.415000e+10,9.999000e+03,9.090000e+02,2.360000e+02,5.934000e+05,1.980000e+02,4.353772e+07


Date mutation                20102739
Nature mutation              20102739
Valeur fonciere              19908349
No voie                      12628228
B/T/Q                          905080
Voie                         19951476
Code postal                  20102739
Commune                      20102739
Code departement             20102739
Code commune                 20102739
Nombre de lots               20102739
Type local                   11890422
Surface reelle bati          11877616
Nombre pieces principales    11877616
Surface terrain              13706037
dtype: int64

### **RENAME COLUMNS**

In [12]:
import pandas as pd

# ---------------------------------------------
# Define the Renaming Dictionary
# ---------------------------------------------
rename_columns = {
    "Date mutation": "transaction_date",
    "Nature mutation": "transaction_type",
    "Valeur fonciere": "property_value",
    "No voie": "street_number",
    "B/T/Q": "btq_code",
    "Voie": "street_name",
    "Code postal": "postal_code",
    "Commune": "com_name",
    "Nombre de lots": "number_of_lots",
    "Type local": "property_type",
    "Surface reelle bati": "building_real_surface",
    "Nombre pieces principales": "main_rooms_count",
    "Surface terrain": "land_surface",
    "Code commune" : "com_code",
    "Code departement" : "dep_code",
    "Code voie" : "street_code"
}

# ---------------------------------------------
# Rename columns using the rename Dictionary
# ---------------------------------------------
df.rename(columns=rename_columns, inplace=True)

# --------------------------------
# Save dataset in a new CSV file
# --------------------------------
df.to_csv("../data/processed/INTERIM_01_STANDARD_VF.csv", index=False)
print(f"✅ Dataset standardized and saved at ../data/processed/INTERIM_01_STANDARD_VF.csv")

KeyboardInterrupt: 

<hr>

## 2️⃣ INVALID VALUES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

### **SHAPE & HEAD**

In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/INTERIM_01_STANDARD_VF.csv")
print("Standardized Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns")
display(df.head())

### **INFO & UNIQUE VALUES**

In [ ]:
# Display standardized column names
print("Standardized Column Names:")
display(df.info())

# Loop through each column and print unique values
print(100 * "-")
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")

---- FINAL VALEURS FONCIERES CSV TABLE ------------------------
 //   Column                 Dtype  
---  ------                 -----  
// generate
-1   transaction_id         object (gotta generate it somehow)
// clean
 0   transaction_date       object 
 1   transaction_type       object 
 2   property_value         float64 
 3   street_number          object
 4   btq_code               object  
 5   street_name            object  
 6   postal_code            object  
 7   com_name               object  
 8   number_of_lots         int64   
 9   property_type          object  
 10  building_real_surface  float64 
 11  main_rooms_count       int64

// add 
 12  insee_code             object
 13  com_type               object 
 14  dep_code               object
 15  dep_name          	    object
// calculate  
16  value_per_m2           object


---
### **CREATE `df_com`**
- create df_com with select columns from v_communes_2026.csv
	- COM
	- TYPECOM
	- NCCENR (uppercase + encoding & decoding)
	- DEP
- rename df_com columns
	- insee_code = COM
	- com_type = TYPECOM 
	- com_name = NCCENR
	- dep_code = DEP


In [ ]:
#create df_com with select columns from v_communes_2026.csv
df_com = 
    - COM
	- TYPECOM
	- NCCENR (uppercase + encoding & decoding)
	- DEP
- rename df_com columns
	- insee_code = COM
	- com_type = TYPECOM 
	- com_name = NCCENR
	- dep_code = DEP

In [ ]:
# load raw communes dataset
try:
    df_com = pd.read_csv("../data/raw/v_commune_2026.csv", sep=",", encoding="utf-8")
except UnicodeDecodeError:
    df_com = pd.read_csv("../data/raw/v_commune_2026.csv", sep=",", encoding="latin-1")

# rename df_com columns
df_com.rename(columns={
    "COM": "insee_code",
    "TYPECOM": "com_type",
    "NCCENR": "com_name",
    "DEP": "dep_code"
}, inplace=True)

# uppercase + encoding & decoding of com_name
df_com["com_name"] = (
    df_com["com_name"]
    .astype(str)
    .str.upper()
)

# drop other columns except insee_code, com_type, com_name, dep_code
df_com = df_com[["insee_code", "com_type", "com_name", "dep_code"]]

# display df_com
print("Communes Dataset Shape:", df_com.shape[0], "rows and", df_com.shape[1], "columns")
display(df_com.head())
print(100 * "-")
# display unique values of com_name
print("Unique Values of `com_name`:")
display(df_com["com_name"].unique())

# save df_com in a new CSV file
df_com.to_csv("../data/processed/COM_2026.csv", index=False)

#### **CREATE `df_dep`**

In [ ]:
# load raw departement dataset
try:
    df_dep = pd.read_csv("../data/raw/v_departement_2026.csv", sep=",", encoding="utf-8")
except UnicodeDecodeError:
    df_dep = pd.read_csv("../data/raw/v_departement_2026.csv", sep=",", encoding="latin-1")

# rename df_dep columns
df_dep.rename(columns={
    "DEP": "dep_code",
    "NCCENR": "dep_name"
}, inplace=True)

# uppercase + encoding & decoding of dep_name
df_dep["dep_name"] = (
    df_dep["dep_name"]
    .astype(str)
    .str.upper()
)

# drop other columns except dep_code, dep_name
df_dep = df_dep[["dep_code", "dep_name"]]

# display df_dep
print("Departements Dataset Shape:", df_dep.shape[0], "rows and", df_dep.shape[1], "columns")
display(df_dep.head())
print(100 * "-")
# display unique values of dep_name
print("Unique Values of `dep_name`:")
display(df_dep["dep_name"].unique())

# save df_dep in a new CSV file
df_dep.to_csv("../data/processed/DEP_2026.csv", index=False)

---
from `df_com` :
- ✔ COMTYPE → commune type
- ✔ COM → commune INSEE code
- ✔ NCCENR → official commune names in MAJ & "-" & ACCENT
- ✔ DEP → department code

from `df_dep` :
 - ✔ DEP → department code
 - ✔ NCCENR → official department names NAMES

combine: add dep_name using DEP

rename columns

save csv


###  **CREATE `df_com_dep`**

**USE `com_name`**

In [ ]:
# create df_com_dep by merging df_com and df_dep on dep_code
df_com_dep = pd.merge(df_com, df_dep, on="dep_code", how="left")
# display df_com_dep
print("Combined Communes & Departements Dataset Shape:", df_com_dep.shape[0], "rows and", df_com_dep.shape[1], "columns")
display(df_com_dep.head())
print(100 * "-")
# display unique values of com_name and dep_name in df_com_dep
print("Unique Values of `com_name` in df_com_dep:")
display(df_com_dep["com_name"].unique())
print("Unique Values of `dep_name` in df_com_dep:")
display(df_com_dep["dep_name"].unique())
# save df_com_dep in a new CSV file
df_com_dep.to_csv("../data/processed/COM_DEP_2026.csv", index=False)

---
### **DEAL with INVALID VALUES**

#### *Using "com_name" from `df` & `df_com_dep`*

Column: `com_name`
- df: 
    - MAJ
    - "-"
    - accent
    - some incorrect spelling from official name

- df_com_dep:
    - correct spelling
    - MAJ
    - "-"
    - accent





#### fix df["com_name"] using df_com_dep["com_name"]

In [ ]:
! pip install rapidfuzz

### MATCHING COM_NAME in DF

In [ ]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

# load the combined communes & departements dataset
df_com_dep = pd.read_csv("../data/processed/COM_DEP_2026.csv")

THRESHOLD = 90

# -------------------------
# NORMALIZATION FUNCTION
# -------------------------
def normalize_name(name):
    if pd.isna(name):
        return name
    
    name = str(name)
    
    # remove parentheses (e.g. "(LA)")
    name = re.sub(r"\(.*?\)", "", name)
    
    # normalize spaces
    name = re.sub(r"\s+", " ", name)
    
    return name.strip()

# -------------------------
# PREPARE DATA
# -------------------------
df["com_clean"] = df["com_name"].apply(normalize_name)
df_com_dep["com_clean"] = df_com_dep["com_name"].apply(normalize_name)

official_names = df_com_dep["com_clean"].dropna().unique()
official_set = set(official_names)

unique_names = df["com_clean"].dropna().unique()

# -------------------------
# BUILD MATCHING (small scale = fast)
# -------------------------
mapping_clean = {}

# exact matches
for name in unique_names:
    if name in official_set:
        mapping_clean[name] = name

# fuzzy for remaining
unmatched = [n for n in unique_names if n not in mapping_clean]

for name in unmatched:
    match, score, _ = process.extractOne(
        name,
        official_names,
        scorer=fuzz.token_sort_ratio
    )
    
    if score >= THRESHOLD:
        mapping_clean[name] = match
    else:
        mapping_clean[name] = None  # optional

# -------------------------
# FINAL MAPPING: noisy → official spelling
# -------------------------
clean_to_official = dict(
    zip(df_com_dep["com_clean"], df_com_dep["com_name"])
)

final_mapping = {
    noisy: clean_to_official.get(clean)
    for noisy, clean in mapping_clean.items()
}

# -------------------------
# APPLY (NO MERGE)
# -------------------------
df["com_name"] = df["com_clean"].map(final_mapping)

# -------------------------
# CLEANUP
# -------------------------
df.drop(columns=["com_clean"], inplace=True)

# -------------------------
# CHECK
# -------------------------
print("Unmatched:", df["com_name"].isna().sum())

In [ ]:
# display nulls count in each column
print("Null Values Count in Each Column:")
display(df.isna().sum())


# use postal_code column to generate missing com_name where null, using df_com_dep mapping
postal_to_com = df_com_dep.set_index("postal_code")["com_name"].to_dict()
df["com_name"] = df.apply(lambda row: postal_to_com.get(row["postal_code"], row["com_name"]) if pd.isna(row["com_name"]) else row["com_name"], axis=1)


---
#### **(postal_code, street_name) -> most frequent known com_name**

In [ ]:
import pandas as pd

# -------------------------
# 1. KEEP ONLY KNOWN ROWS
# -------------------------
known = df.loc[
    df["com_name"].notna() &
    df["postal_code"].notna() &
    df["street_name"].notna(),
    ["postal_code", "street_name", "com_name"]
].drop_duplicates()

# -------------------------
# 2. FIND UNIQUE (postal_code, street_name) -> com_name
# -------------------------
unique_pairs = (
    known.groupby(["postal_code", "street_name"])["com_name"]
    .nunique()
    .reset_index(name="n_com")
)

unique_pairs = unique_pairs[unique_pairs["n_com"] == 1]

# -------------------------
# 3. BUILD STRICT MAPPING
# -------------------------
strict_mapping = (
    known.merge(
        unique_pairs[["postal_code", "street_name"]],
        on=["postal_code", "street_name"],
        how="inner"
    )
    .drop_duplicates(["postal_code", "street_name"])
    .set_index(["postal_code", "street_name"])["com_name"]
)

# -------------------------
# 4. FILL ONLY NULL com_name
# -------------------------
mask = (
    df["com_name"].isna() &
    df["postal_code"].notna() &
    df["street_name"].notna()
)

lookup_index = pd.MultiIndex.from_frame(
    df.loc[mask, ["postal_code", "street_name"]]
)

df.loc[mask, "com_name"] = strict_mapping.reindex(lookup_index).to_numpy()

# -------------------------
# 5. CHECK
# -------------------------
print("Remaining NULL com_name:", df["com_name"].isna().sum())

---
### verifying how loading works with accents

In [ ]:
import pandas as pd
# load the combined communes & departements dataset
df_com_dep = pd.read_csv("../data/processed/COM_DEP_2026.csv")
# display df_com_dep
print("Combined Communes & Departements Dataset Shape:", df_com_dep.shape[0], "rows and", df_com_dep.shape[1], "columns")
display(df_com_dep.head())
# unique values of com_name and dep_name in df_com_dep
print("Unique Values of `com_name` in df_com_dep:")
display(df_com_dep["com_name"].unique())

In [ ]:
import pandas as pd
# load the combined communes & departements dataset
df_com_dep = pd.read_csv("../data/processed/COM_DEP_2026.csv")
df = pd.read_csv("../data/processed/INTERIM_01_STANDARD_VF.csv")

# fix df["com_name"] using df_com_dep["com_name"]
df = df.merge(df_com_dep[["com_name", "dep_name"]], on="com_name", how="left")
# in case of inconsistencies in com_name, we will have some null values in dep_name after the merge
# display df after merge
print("Dataset after merging with combined communes & departements:")
display(df.head())
# unique values df["com_name"] and df["dep_name"] after merge
print("Unique Values of `com_name` in df after merge:")
display(df["com_name"].unique())
print("Unique Values of `dep_name` in df after merge:")

---
#### *Fix `dep_code` in **df***

- Column: `dep_code`
    - df: 
        - 01 to 96 for metropolean departments
        - 97 & 98 for outer-mer departments

    - df_com_dep:
        - 01 to 95 for metropolean departments
        - 96 is missing, and there is nan
        - 97 and no 98 anymore

#### *Add `dep_name` using `dep_code`*
After fixing `dep_code` inconsistency, use **df_com_dep** to add `dep_name` to **df**

#### *Add `insee_code` & `com_type` using **clean** `com_name`**
After fixing `com_name` inconsistency, use **df_com_dep** to add `dep_name` to **df**

In [ ]:

'''
ADD COLUMNS using clean `com_name` in **df** & **df_com_dep**:

- code_insee, # COM is code_insee from v_commune_2026.csv
- com_name # fixed in the previous step existing in VF dataset
- com_type, # TYPECOM from v_commune_2026.csv

- dep_code, # DEP from v_departement_2026.csv
- dep_name, # NCC from v_departement_2026.csv

'''

#### *Fix Other columns*

In [ ]:
# --------------------------------------------------------------------------------------
# Column: transaction_type
# --------------------------------------------------------------------------------------
# fix transaction_type encoding issues to ensure consistent values
# replace values to english
df['transaction_type'] = df['transaction_type'].replace({
        "Vente": "Sale",
        "Vente en l'Ã©tat futur d'achÃ¨vement": "Sale in future state of completion",
        "Vente terrain Ã\xa0 bÃ¢tir": "Sale of unbuilt land",
        "Echange": "Exchange",
        "Adjudication": "Auction",
        "Expropriation": "Expropriation"})

# --------------------------------------------------------------------------------------
# Column: btq_code
# --------------------------------------------------------------------------------------
# fix btq_code inconsistencies: replace invalid values with NaN and standardize 'b' to 'B'
invalid_values = ['-', '.', '*']
df["btq_code"] = df["btq_code"].replace(invalid_values, pd.NA)
df['btq_code'] = df['btq_code'].replace({ 'b': 'B'})

# --------------------------------------------------------------------------------------
# Column: building_real_surface
# --------------------------------------------------------------------------------------
# because it is probably not a built property if building_real_surface is NaN
# as building area is not relevant for land sales
# fill NaN with 0 
df["building_real_surface"] = df["building_real_surface"].fillna(0)

# --------------------------------------------------------------------------------------
# Column: main_rooms_count
# --------------------------------------------------------------------------------------
# because it is probably not a built property if main_rooms_count is NaN
# fill NaN with 0 
df["main_rooms_count"] = df["main_rooms_count"].fillna(0)

# --------------------------------------------------------------------------------------
# Column: surface_type
# --------------------------------------------------------------------------------------
# surface_type: replace 'bati' with 'built' and 'terrain' with 'land'
df['surface_type'] = df['surface_type'].replace({
    "bati": "built",    
    "terrain": "land"})


# --------------------------------------------------------------------------------------------
# SAVE - InvalidValues ValeursFoncieres
# --------------------------------------------------------------------------------------------
df.to_csv("../data/processed/INTERIM_02_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with invalid values handled and saved at ../data/processed/INTERIM_02_ValeursFoncieres.csv")

<hr>

## 3️⃣ DATA TYPES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

### **SHAPE & HEAD**

In [ ]:
# Load the dataset
df = pd.read_csv("../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv")

# Display DataFrame
print("DataFrame:")
print(df.shape[0], "rows and", df.shape[1], "columns")
display(df.head())

### **DATA TYPES**

In [ ]:
# print data types and unique values count for each column in df
print("Data types - Valeurs Foncieres:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

### **UNIQUE VALUES**

In [ ]:
# Loop through each column and print unique values
print(100 * "-")
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")

| Field Name                                 | Python data type             |
|--------------------------------------------|------------------------------|
| **transaction_date**                          | `object` to `date`??           |
| **street_number**                                | `float64` to `object`         |
| **postal_code**                            | `float64` to `object`        |
| **main_rooms**              | `float64` to `int64`         |


### **CONVERT DATA TYPES**

In [ ]:
# convert df columns types as follows:
#df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors='coerce')
df["street_number"] = df["street_number"].astype('object')
df["postal_code"] = df["postal_code"].astype('object')
df["main_rooms_count"] = df["main_rooms_count"].astype('int64')

# Display Converted column names
print("Converted column names:")
display(df.info())

# Save dataset in a new CSV file
df.to_csv("../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset converted to appropriate data types and saved at ../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv")

<hr>

## 4️⃣ NULL VALUES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

### **INFO - `df_VF`**

In [ ]:
import pandas as pd
# Load dataset with converted data types
df = pd.read_csv("../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv")

# display data types and missing values of each column in df
df.info()

### **NULLS COUNT - `df_VF` (BEFORE)**

In [ ]:
import pandas as pd
# Loop through each column and print the number of nulls in each column
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"[{col}]: [{null_count}] null values")
print(100*"-")

### **DEAL with NULLS - `df_VF`**

In [ ]:
# --------------------------------------------------------------------------------
# Column: property_value (euros)
# --------------------------------------------------------------------------------
# drop nulls in property_value column as it's a critical column for analysis
df = df.dropna(subset=["property_value"])

# --------------------------------------------------------------------------------
# Column: street_number
# --------------------------------------------------------------------------------
# nulls in street_number are common and not necessarily invalid
# Fill NaN with "Unknown" for now
df["street_number"] = df["street_number"].fillna("Unknown")

# --------------------------------------------------------------------------------
# Column: building_real_surface
# --------------------------------------------------------------------------------
# Fill NaN with 0
df["building_real_surface"] = df["building_real_surface"].fillna(0)

# --------------------------------------------------------------------------------
# Column: main_rooms_count
# --------------------------------------------------------------------------------
# main_rooms_count is only relevant for houses/apartments
# fill NaN with 0 when property_type is land
df["main_rooms_count"] = df.apply(lambda row: 0 if pd.isna(row["main_rooms_count"]) and row["property_type"] == "land" else row["main_rooms_count"], axis=1)

# --------------------------------------------------------------------------------
# Column: land_surface
# --------------------------------------------------------------------------------
# Fill NaN with 0 if building_real_surface is not null
df["land_surface"] = df.apply(lambda row: 0 if pd.isna(row["land_surface"]) and not pd.isna(row["building_real_surface"]) else row["land_surface"], axis=1)

# --------------------------------------------------------------------------------
# Column: property_type
# --------------------------------------------------------------------------------
# Fill NaN with "Unknown" to keep all rows for analysis, 
# as property_type is important for analysis and we don't want to drop rows with missing property_type
df["property_type"] = df["property_type"].fillna("Unknown")

# --------------------------------------------------------------------------------
# Column: btq_code
# --------------------------------------------------------------------------------
df["btq_code"] = df["btq_code"].fillna("Unknown")

# --------------------------------------------------------------------------------
# Column: street_name
# --------------------------------------------------------------------------------
df["street_name"] = df["street_name"].fillna("Unknown")

# --------------------------------------------------------------------------------
# Column: postal_code
# --------------------------------------------------------------------------------
df["postal_code"] = df["postal_code"].fillna("Unknown")

# --------------------------------------------------------------------------------
# Save dataset in a new CSV file
# --------------------------------------------------------------------------------
df.to_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with null values handled and saved at ../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")

### **NULLS COUNT - `df_VF` (AFTER)**

In [ ]:
import pandas as pd

# Load dataset with nulls handled
df = pd.read_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")
print(100*"-")

# display data types and missing values of each column in df
print("Data types and missing values after handling nulls:")
df.info()
print(100*"-")

# Loop through each column and print the number of nulls in each column
print("Null values count after handling nulls:")
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"[{col}]: [{null_count}] null values")
print(100*"-")

<hr>

## 5️⃣ DUPLICATES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

### **DUPLICATE ROWS - `df_VF`**

In [ ]:
import pandas as pd

# load dataset with nulls handled
df = pd.read_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")

# Check for duplicate rows in the DataFrame
duplicate_rows = df[df.duplicated()]

# Print the number of duplicate rows
print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")

# Display the duplicate rows if any exist
if not duplicate_rows.empty:
    print("\nDuplicate rows:")
    display(duplicate_rows)
else:
    print("\nNo duplicate  rows found.")

In [ ]:
print(100*"-")
# calculate percentages nulls in df columns ##.##%
print(f"Nulls percentage in each column:\n{df.isnull().mean() * 100}")
print(100*"-")
# calculate duplicate rows percentage in df
print(f"Duplicate rows percentage: {df.duplicated().mean() * 100:.2f}%")
print(100*"-")

### **DEAL with DUPLICATE ROWS - `df_VF`**

In [ ]:
# drop duplicate rows in df
#df = df.drop_duplicates()
'''
I decided to keep duplicates for now as they represent real transactions 
and we don't want to lose any data at this stage, 
but we can always revisit this decision later if needed.
'''

# Save dataset in a new CSV file
df.to_csv("../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with duplicates handled and saved at ../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv")

<hr>

## 6️⃣ OUTLIERS


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

### **OUTLIERS - `df_VF`**

### **DEAL with OUTLIERS - `df_VF`**

<hr>

## 🕵️ REVISE & SAVE CLEAN DATA


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

```text
- display shape
- display head
- display info
- display unique values
- check data types
- check nulls
- check duplicates
```

In [ ]:
import pandas as pd

# --------------------------------------------
# Load dataset with duplicates handled
# --------------------------------------------
df = pd.read_csv("../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv")
print(100*"-")

# --------------------
# 🔲 DISPLAY - HEAD
# --------------------
# Display df DataFrame
print("Data Revision:")
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")
display(df.head())
print(100*"-")

# --------------------
# 🔲 DISPLAY - INFO
# --------------------
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())
print(100*"-")

# -----------------------------
# 🕵️ CHECK - UNIQUE VALUES
# -----------------------------

# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")
print(100*"-")

# -----------------------------
# 🕵️ CHECK - DATA TYPES
# -----------------------------
print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")
print(100*"-")

# -----------------------------
# 🕵️ CHECK - NULLS
# -----------------------------
# Loop through each column and print the number of nulls in each column
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"[{col}]: [{null_count}] null values")
print(100*"-")


# -----------------------------
# 🕵️ CHECK - DUPLICATES
# -----------------------------
# Check for duplicate rows in the DataFrame
duplicate_rows = df[df.duplicated()]

# Print the number of duplicate rows
print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")
print(f"Duplicate rows percentage: {df.duplicated().mean() * 100:.2f}%")
print(100*"-")

# --------------------------------------------
# Save CLEAN data in a new CSV file
# --------------------------------------------
df.to_csv("../data/processed/CLEAN_ValeursFoncieres_2020-S2_2025-S1.csv", index=False)
print(f"✅ Dataset cleaned and saved at ../data/processed/CLEAN_ValeursFoncieres_2020-S2_2025-S1.csv")

### ADD LATER in revision optimize dataset

In [ ]:
## TO ADD LATER in my table in one column and drop value and surface reel & terain 

    "surface_used": "effective_surface",
    "Prix_m2": "value_per_m2",
    "surface_type": "surface_type",
    "departement": "department_code"

# just leave: price per m² + department code + surface type (built/land)
            # -------------------------
            # PRICE PER M²
            # -------------------------
            chunk["value_per_m2"] = chunk["Valeur fonciere"] / chunk["effective_surface"]

            # -------------------------
            # DEPARTMENT CODE
            # -------------------------
            # maybe use the INSEE TABLEAU DE LA SOCIETE for more accuracy
            chunk["department_code"] = chunk["Code postal"].astype(str).str[:2]
